# Sesión 4 -> Interpretar como ingeniero.

1)	¿Qué porcentaje de muestras es de cada NetworkMode? ¿Qué % de cada clase de cobertura hay en Driving frente a Static?


2)	Cuando NetworkMode es 5G, existen tanto RSRP como NRxRSRP. 🧠 Antes de investigar: ¿qué crees que mide cada una? Pista: piensa en cómo se desplegó 5G comercial en 2019–2020 (NSA, EN-DC). Después busca qué dice el paper sobre esas columnas.


3)	¿En qué unidad está DL_bitrate? Búscalo en el paper (Raca et al., MMSys 2020) y corrige la etiqueta "kbps*" de la app si hace falta.

4)	Encuentra un tramo en el mapa con RSRP bueno pero SNR malo. ¿Qué explicación radio tiene?


In [3]:
''' 1) '''
import pandas as pd
import sys

sys.path.append("../..")

from src import loader

ruta = "../../data/raw/5G-production-dataset"
df = loader.cargar_todos(ruta)
df.head() ## Dataframe cargado

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,...,PINGLOSS,CELLHEX,NODEHEX,LACHEX,RAWCELLID,NRxRSRP,NRxRSRQ,servicio,movilidad,fichero
0,2019-11-28 07:27:57,-8.388193,51.935608,0,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
1,2019-11-28 07:27:57,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
2,2019-11-28 07:27:58,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
3,2019-11-28 07:27:59,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,3.0,...,NaN,C,A81B,9CBA,11016972,-101.0,-3.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
4,2019-11-28 07:28:00,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,3.0,...,NaN,C,A81B,9CBA,11016972,-101.0,-3.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv


In [ ]:
df["NetworkMode"].value_counts(normalize=True)*100 ## Porcentaje de muestras por NetworkMode.

NetworkMode
5G       72.281425
LTE      18.659220
HSPA+     8.414984
UMTS      0.260716
HSUPA     0.249058
HSDPA     0.086375
GPRS      0.041333
EDGE      0.006889
Name: proportion, dtype: float64

In [ ]:

mascara_driv = df["movilidad"] == "Driving" ## Ponemos la máscara que debe cumplirse
RSRP_driving = df.loc[mascara_driv,"RSRP"] ## Sacamos los valores con .loc[filas, columnas]

from src import kpis ## Para hallar la calidad de cobertura de acuerdo a una Serie de RSRPs

print("Porcentaje de calidad de muestras RSRP en Driving:")
print((kpis.clasificar_cobertura(RSRP_driving).value_counts(normalize=True)*100).round(2))

## Hacemos lo mismo con Static
mascara_stat = df["movilidad"] == "Static" 
RSRP_stat = df.loc[mascara_stat,"RSRP"] 
print("Porcentaje de calidad de muestras RSRP en Static:")
print((kpis.clasificar_cobertura(RSRP_stat).value_counts(normalize=True)*100).round(2))

Porcentaje de calidad de muestras RSRP en Driving:
RSRP
Excelente    36.09
Buena        27.20
Regular      25.17
Mala         11.55
Name: proportion, dtype: float64
Porcentaje de calidad de muestras RSRP en Static:
RSRP
Regular      56.93
Mala         24.02
Buena        13.05
Excelente     6.00
Name: proportion, dtype: float64


In [38]:
'''Cuando NetworkMode es 5G, existen tanto RSRP como NRxRSRP. 🧠 Antes de investigar: ¿qué crees que mide cada una? Pista: piensa en cómo se desplegó 5G comercial en 2019–2020 (NSA, EN-DC). Después busca qué dice el paper sobre esas columnas.'''
mascara = df["NetworkMode"] == "5G"
df.loc[mascara,["RSRP","NRxRSRP"]].value_counts()

RSRP    NRxRSRP
-100.0  -104.0     1859
-99.0   -101.0     1855
-100.0  -103.0     1828
-99.0   -102.0     1794
        -103.0     1614
                   ... 
-63.0   -62.0         1
-112.0  -91.0         1
-109.0  -70.0         1
-70.0   -101.0        1
-77.0   -106.0        1
Name: count, Length: 2505, dtype: int64

In [45]:
'''Encuentra un tramo en el mapa con RSRP bueno pero SNR malo. ¿Qué explicación radio tiene?'''

mascara = (df["RSRP"] > -80) & (df["SNR"] < -10) ## Ponemos un RSRP bueno -> mayor a -80 dBm. Y ponemos un SNR malo -> menor a -10 dB
df.loc[mascara, ["RSRP","NRxRSRP"]].head(10) ## Cogemos la longitud y latitud de uno de estos puntos. Ej: -8.481600, 51.897247

,RSRP,NRxRSRP
1528,-75.0,-69.0
1529,-75.0,-69.0
2102,-76.0,-84.0
2103,-76.0,-84.0
2107,-77.0,-84.0
2108,-77.0,-84.0
4890,-78.0,-88.0
4891,-78.0,-88.0
4892,-78.0,-88.0
4895,-79.0,-88.0
